In [1]:
import numpy as np
import pandas as pd
from ucimlrepo import fetch_ucirepo 
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [2]:
# 1. Cargar dataset
census_income = fetch_ucirepo(id=20) 
X = census_income.data.features
y = census_income.data.targets.values.ravel()

# Preprocesar
X = X.fillna("Unknown")
for col in X.columns:
    if X[col].dtype == "object":
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [3]:
# 2. Active learning: dividir en etiquetados y no etiquetados
n_initial = 50  # etiquetas iniciales
labeled_idx = np.random.choice(len(X_train), n_initial, replace=False)
unlabeled_idx = np.array([i for i in range(len(X_train)) if i not in labeled_idx])

X_labeled, y_labeled = X_train[labeled_idx], y_train[labeled_idx]
X_unlabeled, y_unlabeled = X_train[unlabeled_idx], y_train[unlabeled_idx]   

In [4]:
# 3. Bucle de aprendizaje activo
model = LogisticRegression(max_iter=5000)
performance = []

for step in range(10):  # 10 rondas de aprendizaje activo
    # Entrenar con lo que tenemos etiquetado
    model.fit(X_labeled, y_labeled)
    
    # Evaluar en test
    acc = accuracy_score(y_test, model.predict(X_test))
    performance.append(acc)
    print(f"Iter {step+1} - Precisión en test: {acc:.4f} con {len(y_labeled)} muestras")
    
    # Si ya no quedan datos sin etiquetar, rompemos
    if len(X_unlabeled) == 0:
        break

Iter 1 - Precisión en test: 0.4815 con 50 muestras
Iter 2 - Precisión en test: 0.4815 con 50 muestras
Iter 3 - Precisión en test: 0.4815 con 50 muestras
Iter 4 - Precisión en test: 0.4815 con 50 muestras
Iter 5 - Precisión en test: 0.4815 con 50 muestras
Iter 6 - Precisión en test: 0.4815 con 50 muestras
Iter 7 - Precisión en test: 0.4815 con 50 muestras
Iter 8 - Precisión en test: 0.4815 con 50 muestras
Iter 9 - Precisión en test: 0.4815 con 50 muestras
Iter 10 - Precisión en test: 0.4815 con 50 muestras


In [6]:
  # 4. Calcular incertidumbre (muestras con prob más cercana a 0.5)
probs = model.predict_proba(X_unlabeled)
uncertainty = np.abs(probs[:,1] - 0.5)  # más pequeño = más incierto
query_idx = np.argsort(uncertainty)[:20]  # seleccionar 20 más inciertos

In [7]:
    # 5. Simular oráculo: obtener etiquetas verdaderas
X_new, y_new = X_unlabeled[query_idx], y_unlabeled[query_idx]

In [8]:
# 6. Mover a conjunto etiquetado
X_labeled = np.vstack([X_labeled, X_new])
y_labeled = np.hstack([y_labeled, y_new])
    
mask = np.ones(len(X_unlabeled), dtype=bool)
mask[query_idx] = False
X_unlabeled, y_unlabeled = X_unlabeled[mask], y_unlabeled[mask]